# PS LiDAR - Laboratorio de Desarrollo

**Ladrillos disponibles:**
- Brick 1: Carga de Datos
- Brick 2: Recorte Circular (coordenadas manuales)
- **Brick 3.5: Filtrado de Ruido (SOR)** ← NUEVO
- Brick 3: Detección de Normalización
- Brick 4: Filtrado de Suelo
- Brick 5: Normalización de Altura
- Brick 5.5: Exportar Checkpoints
- Brick 6: Visualización 3D
- Brick 7: Segmentación de Árboles
- **Brick 7b: Separación de Understory** ← NUEVO

In [1]:
import os
import sys
import time
from pathlib import Path

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.core import (
    PointCloudLoader,
    detect_normalization,
    classify_ground,
    clip_circular_plot,
    normalize_heights,
    export_point_cloud,
    segment_trees,
    filter_noise_sor,
    separate_understory,
)

print("✓ Módulos importados")

✓ Módulos importados


---
## 1. Cargar Archivo (Brick 1)

In [3]:
FILE_PATH = "C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw/HQP079_01_raw.laz"

loader = PointCloudLoader(FILE_PATH)
loader.load()

meta = loader.get_metadata()
print(f"Archivo: {meta['filename']}")
print(f"Puntos: {meta['point_count']:,}")
print(f"Tamaño: {meta['file_size_mb']} MB")

Archivo: HQP079_01_raw.laz
Puntos: 19,173,291
Tamaño: 209.63 MB


In [4]:
# Cargar XYZ y campos escalares disponibles
xyz_full = loader.get_xyz()

scalar_fields = {}
for field in ['intensity', 'return_number', 'number_of_returns', 'classification']:
    try:
        scalar_fields[field] = loader.get_attribute(field)
        print(f"✓ {field}: {len(scalar_fields[field]):,} valores")
    except:
        print(f"⚠ {field}: no disponible")

print(f"\nMemoria XYZ: {xyz_full.nbytes / (1024**2):.1f} MB")

✓ intensity: 19,173,291 valores
✓ return_number: 19,173,291 valores
✓ number_of_returns: 19,173,291 valores
✓ classification: 19,173,291 valores

Memoria XYZ: 438.8 MB


---
## 2. Recorte Circular (Brick 2)

**Instrucciones:**
1. Abrir el archivo original en CloudCompare
2. Usar herramienta "Point Picking" para ubicar el centro del mat
3. Copiar las coordenadas Xg, Yg mostradas
4. Pegar los valores en `CENTER_X` y `CENTER_Y` abajo

In [5]:
# ═══════════════════════════════════════════════════════════════
# PARÁMETROS DEL USUARIO - Modificar según el plot
# ═══════════════════════════════════════════════════════════════

# Coordenadas del centro (obtenidas de CloudCompare Point Picking)
CENTER_X = -0.311372
CENTER_Y = -1.461118

# Radio del plot en metros
PLOT_RADIUS = 16.0

# ═══════════════════════════════════════════════════════════════

print(f"Centro: ({CENTER_X:.6f}, {CENTER_Y:.6f})")
print(f"Radio: {PLOT_RADIUS}m")

Centro: (-0.311372, -1.461118)
Radio: 16.0m


In [6]:
# Ejecutar recorte circular
t0 = time.perf_counter()
clip_result = clip_circular_plot(xyz_full, CENTER_X, CENTER_Y, PLOT_RADIUS)
elapsed = time.perf_counter() - t0

plot_indices = clip_result.indices

print(f"✓ Recorte en {elapsed*1000:.0f}ms")
print(f"Puntos originales: {len(xyz_full):,}")
print(f"Puntos en plot: {clip_result.n_points:,} ({clip_result.n_points/len(xyz_full):.1%})")

✓ Recorte en 1017ms
Puntos originales: 19,173,291
Puntos en plot: 10,040,816 (52.4%)


In [7]:
# Aplicar recorte a XYZ y campos escalares
xyz = xyz_full[plot_indices]

plot_scalars = {}
for field, values in scalar_fields.items():
    plot_scalars[field] = values[plot_indices]

print(f"Plot XYZ: {xyz.shape}")
print(f"Campos escalares: {list(plot_scalars.keys())}")

# Liberar memoria
del xyz_full, scalar_fields
import gc; gc.collect()
print("✓ Memoria liberada")

Plot XYZ: (10040816, 3)
Campos escalares: ['intensity', 'return_number', 'number_of_returns', 'classification']
✓ Memoria liberada


---
## 2.5 Filtrado de Ruido (Brick 3.5)

In [8]:
# ═══════════════════════════════════════════════════════════════
# BRICK 3.5: FILTRADO DE RUIDO (SOR)
# ═══════════════════════════════════════════════════════════════

print("Aplicando filtro de ruido SOR...")
t0 = time.perf_counter()
noise_result = filter_noise_sor(xyz, k_neighbors=10, std_ratio=2.0, verbose=True)

# Reemplazar xyz con puntos limpios
xyz = noise_result.clean_xyz
plot_scalars = {k: v[noise_result.clean_indices] for k, v in plot_scalars.items()}

print(f"\n✓ Filtrado en {time.perf_counter()-t0:.1f}s")

Aplicando filtro de ruido SOR...
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
SOR filtering: 10,040,816 points, k=10, std=2.0
  Removed: 432,381 points (4.31%)
  Remaining: 9,608,435 points

✓ Filtrado en 33.4s


---
## 3. Análisis de Normalización (Brick 3)

In [9]:
analysis = detect_normalization(xyz)
print(f"Estatus: {analysis.status.value.upper()}")
print(f"¿Normalizada?: {analysis.is_normalized}")
print(f"Rango Z: {analysis.z_min:.2f}m a {analysis.z_max:.2f}m")

Estatus: NOT_NORMALIZED
¿Normalizada?: False
Rango Z: -2.14m a 22.73m


---
## 4. Filtrado de Suelo (Brick 4)

In [10]:
print("Ejecutando CSF...")
t0 = time.perf_counter()

ground_result = classify_ground(
    xyz,
    cloth_resolution=1.0,
    rigidness=1,
    class_threshold=0.5,
    slope_smooth=True,
)

print(f"✓ Completado en {time.perf_counter() - t0:.2f}s")
print(f"Suelo: {ground_result.n_ground:,} ({ground_result.ground_ratio:.1%})")
print(f"Vegetación: {ground_result.n_off_ground:,}")

Ejecutando CSF...
✓ Completado en 4.21s
Suelo: 2,922,123 (30.4%)
Vegetación: 6,686,312


In [11]:
# Separar suelo y vegetación
ground_xyz = xyz[ground_result.ground_indices]
vegetation_xyz = xyz[ground_result.off_ground_indices]

ground_scalars = {k: v[ground_result.ground_indices] for k, v in plot_scalars.items()}
vegetation_scalars = {k: v[ground_result.off_ground_indices] for k, v in plot_scalars.items()}

print(f"Suelo: {len(ground_xyz):,} puntos")
print(f"Vegetación: {len(vegetation_xyz):,} puntos")

Suelo: 2,922,123 puntos
Vegetación: 6,686,312 puntos


---
## 5. Normalización de Altura (Brick 5)

In [12]:
print("Normalizando alturas...")
t0 = time.perf_counter()

veg_norm_result = normalize_heights(vegetation_xyz, ground_xyz, resolution=0.5)
veg_normalized = veg_norm_result.xyz_normalized

ground_norm_result = normalize_heights(ground_xyz, ground_xyz, resolution=0.5)
ground_normalized = ground_norm_result.xyz_normalized

print(f"✓ Completado en {(time.perf_counter() - t0)*1000:.0f}ms")
print(f"")
print(f"Vegetación: Z = {veg_normalized[:, 2].min():.2f}m a {veg_normalized[:, 2].max():.2f}m")
print(f"Suelo: Z = {ground_normalized[:, 2].min():.2f}m a {ground_normalized[:, 2].max():.2f}m")

Normalizando alturas...
✓ Completado en 1254ms

Vegetación: Z = -0.70m a 25.03m
Suelo: Z = -0.69m a 0.73m


---
## 5.5 Exportar Checkpoints

In [ ]:
# Directorio de salida
OUTPUT_DIR = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw")

# Exportar vegetación
veg_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_veg_normalized.laz")
export_point_cloud(
    veg_file,
    veg_normalized,
    intensity=vegetation_scalars.get('intensity'),
    return_number=vegetation_scalars.get('return_number'),
    number_of_returns=vegetation_scalars.get('number_of_returns'),
    classification=vegetation_scalars.get('classification'),
)
print(f"✓ Vegetación: {veg_file.name} ({veg_file.stat().st_size / (1024**2):.1f} MB)")

# Exportar suelo
ground_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_ground_normalized.laz")
export_point_cloud(
    ground_file,
    ground_normalized,
    intensity=ground_scalars.get('intensity'),
    return_number=ground_scalars.get('return_number'),
    number_of_returns=ground_scalars.get('number_of_returns'),
    classification=ground_scalars.get('classification'),
)
print(f"✓ Suelo: {ground_file.name} ({ground_file.stat().st_size / (1024**2):.1f} MB)")

---
## 6. Visualización 3D (Brick 6)

In [ ]:
import open3d as o3d
import numpy as np

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(veg_normalized)

# Colorear por altura
z = veg_normalized[:, 2]
z_scaled = (z - z.min()) / (z.max() - z.min() + 1e-6)
colors = np.zeros((len(z_scaled), 3))
colors[:, 0] = z_scaled
colors[:, 1] = 1 - np.abs(2 * z_scaled - 1)
colors[:, 2] = 1 - z_scaled
pcd.colors = o3d.utility.Vector3dVector(colors)

print(f"Nube: {len(pcd.points):,} puntos")

In [ ]:
o3d.visualization.draw_geometries([pcd], window_name="Vegetación Normalizada", width=1280, height=720)

---
## 7. Geometrics features extration

---
#### Upload files from ceckpoint

In [1]:
# ═══════════════════════════════════════════════════════════════
# CARGAR DESDE CHECKPOINT (saltar Bricks 1-5.5)
# ═══════════════════════════════════════════════════════════════
import os
import sys
import laspy
import numpy as np
from pathlib import Path

# Agregar módulo al path (CORREGIDO)
project_root = Path("c:/Users/geoal/Documents/SoftwareDev/PS_LiDAR")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

CHECKPOINT_FILE = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_veg_normalized.laz")
las = laspy.read(str(CHECKPOINT_FILE))
veg_normalized = np.column_stack([las.x, las.y, las.z])
print(f"✓ Cargados {len(veg_normalized):,} puntos desde checkpoint")
print(f"  Rango Z: {veg_normalized[:, 2].min():.2f}m a {veg_normalized[:, 2].max():.2f}m")

✓ Cargados 6,686,312 puntos desde checkpoint
  Rango Z: -0.70m a 25.03m


---

In [2]:
# ═══════════════════════════════════════════════════════════════
# FASE 1: EXTRACCIÓN DE FEATURES GEOMÉTRICAS (OPTIMIZADO)
# ═══════════════════════════════════════════════════════════════
from src.core import compute_all_features_fast, classify_understory, validate_tree_connectivity
import time

print("Fase 1: Extracción de features (optimizado)...")
t0 = time.perf_counter()

features, dist_to_ground, dist_to_top = compute_all_features_fast(
    veg_normalized,
    voxel_size=0.1,      # 10cm voxels
    k_neighbors=20,
    #cylinder_radius=0.5,
    verbose=True
)

print(f"✓ Features extraídos en {time.perf_counter() - t0:.1f}s")

Fase 1: Extracción de features (optimizado)...
Computing geometric features (optimized) for 6,686,312 points...
  Voxel size: 0.1m, K-neighbors: 20
  Voxelized: 6,686,312 points → 1,146,011 voxels (17.1%)
  Computing PCA on 1,146,011 voxel centroids...
    Processed 100,000 / 1,146,011 voxels (8.7%)
    Processed 200,000 / 1,146,011 voxels (17.5%)
    Processed 300,000 / 1,146,011 voxels (26.2%)
    Processed 400,000 / 1,146,011 voxels (34.9%)
    Processed 500,000 / 1,146,011 voxels (43.6%)
    Processed 600,000 / 1,146,011 voxels (52.4%)
    Processed 700,000 / 1,146,011 voxels (61.1%)
    Processed 800,000 / 1,146,011 voxels (69.8%)
    Processed 900,000 / 1,146,011 voxels (78.5%)
    Processed 1,000,000 / 1,146,011 voxels (87.3%)
    Processed 1,100,000 / 1,146,011 voxels (96.0%)
  Re-projecting features to 6,686,312 original points...
  ✓ Feature computation complete (optimized)
    Verticality: min=0.00, max=1.00, mean=0.53
    Linearity: min=0.00, max=0.99, mean=0.35
    Spheric

In [3]:
# ═══════════════════════════════════════════════════════════════
# FASE 2: CLASIFICACIÓN DE UNDERSTORY
# ═══════════════════════════════════════════════════════════════
print("\nFase 2: Clasificación de understory...")

classification = classify_understory(
    veg_normalized,
    features.verticality,
    features.linearity,
    features.sphericity,
    dist_to_ground,
    dist_to_top,
    verticality_threshold=0.7,
    sphericity_threshold=0.3,
    max_understory_height=2.0,
    min_canopy_clearance=3.0,  # NUEVO
    verbose=True
)


Fase 2: Clasificación de understory...
Classifying understory: 6,686,312 points
  Thresholds: verticality>0.7, linearity>0.4, sphericity>0.3
  Protection: max_height<2.0m, canopy_clearance>3.0m
  Potential stems: 717,821 points (10.7%)
  Protected (under canopy): 1,305,014 low points
  Understory: 7,713 points (0.1%)
  Tree (including canopy): 6,678,599 points (99.9%)


In [4]:
# ═══════════════════════════════════════════════════════════════
# FASE 3: VALIDACIÓN DE CONECTIVIDAD (OPTIMIZADO)
# ═══════════════════════════════════════════════════════════════
from src.core import validate_tree_connectivity_fast

print("\nFase 3: Validación de conectividad (optimizado)...")

is_valid_tree = validate_tree_connectivity_fast(
    veg_normalized,
    classification.is_stem,
    voxel_size=0.1,
    min_component_size=50,
     min_tree_height=5.0,
    verbose=True
)

veg_for_segmentation = veg_normalized[is_valid_tree]
print(f"\n✓ Puntos para segmentación: {len(veg_for_segmentation):,}")


Fase 3: Validación de conectividad (optimizado)...
Validating tree connectivity (optimized): 6,686,312 points, 717,821 stems
  Voxel size: 0.1m
  Voxelized: 6,686,312 points → 1,146,011 voxels
  Stem voxels: 150,052
  Building voxel adjacency graph...
  Graph: 1,146,011 voxels, 8,786,291 edges
  Found 2,236 connected components
  Filtered: 811 small, 164 short (<5.0m)
  Valid tree points: 6,393,256 (95.6%)
  Disconnected (understory): 293,056 (4.4%)

✓ Puntos para segmentación: 6,393,256


In [5]:
# ═══════════════════════════════════════════════════════════════
# FASE 4: FILTRADO DE UNDERSTORY EN STRIPE
# ═══════════════════════════════════════════════════════════════
from src.core import filter_understory_stripe

print("\nFase 4: Filtrado de understory en stripe...")

is_tree_final = filter_understory_stripe(
    veg_normalized,
    classification.is_stem,
    is_valid_tree,
    stripe_max_height=3.0,  # Ajustable con datos de campo
    verbose=True
)

trees_xyz = veg_normalized[is_tree_final]
understory_xyz = veg_normalized[~is_tree_final]

print(f"\n✓ Árboles: {len(trees_xyz):,} puntos")
print(f"✓ Understory: {len(understory_xyz):,} puntos")


Fase 4: Filtrado de understory en stripe...
Filtering understory in stripe (0 to 3.0m)...
  Points in stripe (<3.0m): 1,603,517
  Stems in stripe (protected): 175,864
  Understory removed from stripe: 1,247,599
  Final: 5,145,657 tree pts (77.0%)
         1,540,655 understory pts (23.0%)

✓ Árboles: 5,145,657 puntos
✓ Understory: 1,540,655 puntos


In [5]:
# ═══════════════════════════════════════════════════════════════
# DIAGNÓSTICO: Analizar componentes conectados
# ═══════════════════════════════════════════════════════════════
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

# Re-calcular conectividad para diagnóstico
voxel_size = 0.1
voxel_indices = np.floor(veg_normalized / voxel_size).astype(np.int32)
unique_voxels, inverse_indices = np.unique(voxel_indices, axis=0, return_inverse=True)

# Build voxel graph (26-connectivity)
voxel_to_idx = {tuple(v): i for i, v in enumerate(unique_voxels)}
rows, cols = [], []
for i, v in enumerate(unique_voxels):
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:
                if dx or dy or dz:
                    n = (v[0]+dx, v[1]+dy, v[2]+dz)
                    if n in voxel_to_idx:
                        rows.append(i); cols.append(voxel_to_idx[n])

adj = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(unique_voxels), len(unique_voxels)))
n_components, labels = connected_components(adj, directed=False)

# Map to points
point_labels = labels[inverse_indices]

# Estadísticas por componente
print(f"Total componentes: {n_components}")
print("\nTop 20 componentes por tamaño:")
for label in np.argsort(np.bincount(point_labels))[::-1][:20]:
    mask = point_labels == label
    count = np.sum(mask)
    z_range = veg_normalized[mask, 2].max() - veg_normalized[mask, 2].min()
    has_stems = np.any(classification.is_stem[mask])
    print(f"  ID {label}: {count:,} pts, Z={z_range:.1f}m, stems={has_stems}")

Total componentes: 2236

Top 20 componentes por tamaño:
  ID 0: 6,232,974 pts, Z=24.7m, stems=True
  ID 564: 160,282 pts, Z=23.0m, stems=True
  ID 1262: 59,398 pts, Z=2.2m, stems=True
  ID 634: 37,009 pts, Z=2.2m, stems=True
  ID 1329: 9,055 pts, Z=1.1m, stems=True
  ID 1: 8,020 pts, Z=2.2m, stems=True
  ID 337: 4,853 pts, Z=1.1m, stems=True
  ID 1407: 4,839 pts, Z=1.0m, stems=True
  ID 148: 4,734 pts, Z=2.4m, stems=True
  ID 1148: 3,942 pts, Z=0.5m, stems=True
  ID 869: 3,917 pts, Z=1.0m, stems=True
  ID 767: 3,792 pts, Z=0.7m, stems=True
  ID 1121: 3,082 pts, Z=2.0m, stems=True
  ID 565: 2,995 pts, Z=1.6m, stems=True
  ID 584: 2,713 pts, Z=0.7m, stems=True
  ID 519: 2,702 pts, Z=0.8m, stems=True
  ID 1668: 2,506 pts, Z=0.5m, stems=True
  ID 1003: 2,431 pts, Z=1.3m, stems=True
  ID 742: 2,423 pts, Z=0.6m, stems=True
  ID 820: 2,163 pts, Z=0.5m, stems=True


---
#### Export checkpoint: Trees & Understory (Separated files)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# EXPORTAR: ÁRBOLES + UNDERSTORY (archivos separados)
# ═══════════════════════════════════════════════════════════════
import laspy

# Usar is_tree_final de Fase 4 (NO is_valid_tree)
trees_xyz = veg_normalized[is_tree_final]
understory_xyz = veg_normalized[~is_tree_final]

# --- Exportar ÁRBOLES ---
trees_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_trees_02.laz")
header = laspy.LasHeader(version="1.4", point_format=0)
las_trees = laspy.LasData(header)
las_trees.x = trees_xyz[:, 0]
las_trees.y = trees_xyz[:, 1]
las_trees.z = trees_xyz[:, 2]

las_trees.add_extra_dim(laspy.ExtraBytesParams(name="verticality", type="float32"))
las_trees.add_extra_dim(laspy.ExtraBytesParams(name="linearity", type="float32"))
las_trees.add_extra_dim(laspy.ExtraBytesParams(name="sphericity", type="float32"))
las_trees.verticality = features.verticality[is_tree_final]
las_trees.linearity = features.linearity[is_tree_final]
las_trees.sphericity = features.sphericity[is_tree_final]

las_trees.write(str(trees_file))
print(f"✓ Árboles: {trees_file.name} ({len(trees_xyz):,} puntos)")

# --- Exportar UNDERSTORY ---
understory_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_understory_02.laz")
header2 = laspy.LasHeader(version="1.4", point_format=0)
las_under = laspy.LasData(header2)
las_under.x = understory_xyz[:, 0]
las_under.y = understory_xyz[:, 1]
las_under.z = understory_xyz[:, 2]

las_under.add_extra_dim(laspy.ExtraBytesParams(name="verticality", type="float32"))
las_under.add_extra_dim(laspy.ExtraBytesParams(name="linearity", type="float32"))
las_under.add_extra_dim(laspy.ExtraBytesParams(name="sphericity", type="float32"))
las_under.verticality = features.verticality[~is_tree_final]
las_under.linearity = features.linearity[~is_tree_final]
las_under.sphericity = features.sphericity[~is_tree_final]

las_under.write(str(understory_file))
print(f"✓ Understory: {understory_file.name} ({len(understory_xyz):,} puntos)")

✓ Árboles: HQP079_01_trees_03.laz (5,145,657 puntos)


ValueError: could not broadcast input array from shape (293056,) into shape (1540655,)

---
---
# BRICK 7: Segmentación de Árboles

**Opción A:** Continuar desde Brick 5 (si ya ejecutaste todo arriba)  
**Opción B:** Cargar checkpoint de vegetación normalizada (si reinicias kernel)

### Opción B: Cargar desde Checkpoint

Ejecuta esta celda SOLO si reiniciaste el kernel y quieres continuar desde el checkpoint.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Instalar pgeof desde dentro del notebook
# ═══════════════════════════════════════════════════════════════

import sys
print(f"Python: {sys.executable}")
!{sys.executable} -m pip install pgeof

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CARGAR DESDE CHECKPOINT (autónomo - incluye todos los imports)
# ═══════════════════════════════════════════════════════════════
import os
import sys
import time
import laspy
import numpy as np
from pathlib import Path

# Agregar módulo al path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.core import segment_trees, separate_understory

CHECKPOINT_FILE = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_veg_filtered_01.laz")

if CHECKPOINT_FILE.exists():
    print(f"Cargando: {CHECKPOINT_FILE.name}")
    las = laspy.read(str(CHECKPOINT_FILE))
    veg_normalized = np.column_stack([las.x, las.y, las.z])
    print(f"✓ Cargados {len(veg_normalized):,} puntos")
    print(f"Rango Z: {veg_normalized[:, 2].min():.2f}m a {veg_normalized[:, 2].max():.2f}m")
    print(f"✓ segment_trees importado")
else:
    print(f"⚠ Archivo no encontrado: {CHECKPOINT_FILE}")
    print("  Ejecuta los Bricks 1-5 primero.")

### 7.1 Segmentación de Árboles

In [24]:
# ═══════════════════════════════════════════════════════════════
# PARÁMETROS DE SEGMENTACIÓN
# ═══════════════════════════════════════════════════════════════

VOXEL_RESOLUTION = 0.05       # Resolución de voxelización (metros)
STRIPE_Z_MIN = 2.0          # Altura mínima para detección de tallos
STRIPE_Z_MAX = 7.0            # Altura máxima para detección de tallos
VERTICALITY_THRESHOLD = 0.7   # Umbral de verticalidad (0-1)
MAX_AXIS_DISTANCE = 2.0       # Distancia máxima al eje para asignación

# ═══════════════════════════════════════════════════════════════

In [ ]:
print("Segmentando árboles...")
t0 = time.perf_counter()

seg_result = segment_trees(
    veg_for_segmentation,
    voxel_resolution=VOXEL_RESOLUTION,
    stripe_z_min=STRIPE_Z_MIN,
    stripe_z_max=STRIPE_Z_MAX,
    verticality_threshold=VERTICALITY_THRESHOLD,
    max_axis_distance=MAX_AXIS_DISTANCE,
    verbose=True
)

elapsed = time.perf_counter() - t0
print(f"\n✓ Completado en {elapsed:.1f}s")
print(f"Árboles detectados: {seg_result.n_trees}")
print(f"Puntos asignados: {len(veg_for_segmentation) - seg_result.unassigned_count:,}")
print(f"Puntos sin asignar: {seg_result.unassigned_count:,}")

In [ ]:
# Resumen por árbol
print("\n=== Resumen por Árbol ===")
print(f"{'ID':>4} {'Puntos':>12} {'Altura Max':>12} {'Desv. Eje':>10}")
print("-" * 42)
for info in seg_result.tree_info:
    print(f"{info.tree_id:>4} {info.n_points:>12,} {info.height_max:>10.1f}m {info.axis_deviation_deg:>9.1f}°")

### 7.2 Exportar con tree_id

In [ ]:
# Exportar nube segmentada con tree_id como campo escalar
import laspy

OUTPUT_DIR = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw")
seg_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_tree_segmented_2.laz")

# Crear archivo LAS
header = laspy.LasHeader(version="1.4", point_format=0)
las_out = laspy.LasData(header)

las_out.x = veg_for_segmentation[:, 0]
las_out.y = veg_for_segmentation[:, 1]
las_out.z = veg_for_segmentation[:, 2]

# Agregar tree_id como campo extra
las_out.add_extra_dim(laspy.ExtraBytesParams(name="tree_id", type="int32", description="Tree ID"))
las_out.tree_id = seg_result.tree_ids

las_out.write(str(seg_file))
print(f"✓ Exportado: {seg_file.name} ({seg_file.stat().st_size / (1024**2):.1f} MB)")
print(f"  Campo escalar 'tree_id' incluido para visualizar en CloudCompare")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.2b CHECKPOINT - Exportar ubicaciones de árboles
# ═══════════════════════════════════════════════════════════════
import os
import sys
import importlib
import numpy as np
import laspy
from pathlib import Path

# Agregar módulo al path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Recargar módulo para obtener versión actualizada
import src.core.segmentation as seg_module
importlib.reload(seg_module)
from src.core.segmentation import export_tree_locations, TreeSegmentationResult, TreeInfo

# Cargar archivo segmentado
SEG_FILE = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_tree_segmented_2.laz")

las = laspy.read(str(SEG_FILE))
xyz = np.column_stack([las.x, las.y, las.z])
tree_ids = las.tree_id

print(f"✓ Cargados {len(xyz):,} puntos")
print(f"  Árboles únicos: {len(np.unique(tree_ids[tree_ids >= 0]))}")

# Reconstruir TreeInfo desde los datos
unique_ids = np.unique(tree_ids[tree_ids >= 0])
tree_info = []
for tid in unique_ids:
    mask = tree_ids == tid
    pts = xyz[mask]
    tree_info.append(TreeInfo(
        tree_id=int(tid),
        centroid=np.mean(pts, axis=0),
        n_points=int(np.sum(mask)),
        height_max=float(np.max(pts[:, 2])),
        height_min=float(np.min(pts[:, 2])),
        axis_direction=np.array([0, 0, 1]),
        axis_deviation_deg=0.0
    ))

seg_result = TreeSegmentationResult(
    tree_ids=tree_ids,
    n_trees=len(tree_info),
    unassigned_count=int(np.sum(tree_ids == -1)),
    tree_info=tree_info
)

# Exportar ubicaciones
locations_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/processed/HQP079_01_tree_locations_2.txt")
tree_locations = export_tree_locations(seg_result, locations_file)

print(f"\n✓ Exportado: {locations_file.name}")
print(f"  {len(tree_locations)} ubicaciones")
print(f"  En CloudCompare: File > Open > ASCII cloud")

### 7.3 Visualización por Árbol (Open3D)

In [ ]:
import open3d as o3d
import numpy as np

# Generar colores únicos por árbol
np.random.seed(42)
n_trees = seg_result.n_trees + 1  # +1 para no asignados
tree_colors = np.random.rand(n_trees, 3)
tree_colors[0] = [0.5, 0.5, 0.5]  # Gris para no asignados (si tree_id == -1)

# Asignar colores
point_colors = np.zeros((len(veg_normalized), 3))
for i, tid in enumerate(seg_result.tree_ids):
    if tid >= 0:
        point_colors[i] = tree_colors[tid + 1]
    else:
        point_colors[i] = tree_colors[0]

# Crear nube
pcd_seg = o3d.geometry.PointCloud()
pcd_seg.points = o3d.utility.Vector3dVector(veg_normalized)
pcd_seg.colors = o3d.utility.Vector3dVector(point_colors)

print(f"Nube segmentada: {len(pcd_seg.points):,} puntos, {seg_result.n_trees} árboles")

In [ ]:
o3d.visualization.draw_geometries([pcd_seg], window_name=f"Segmentación: {seg_result.n_trees} árboles", width=1280, height=720)

---
## 8. Próximos Pasos

- **Brick 8:** Análisis por árbol (DBH, altura, sweep, ramas)
- **Brick 9:** Detección de bifurcaciones
- **Brick 10:** Clasificación HQP de ramas y spikes